# Bagging Model

In [3]:
# 0) Import packages
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier

# 1) Load the data
TRAIN_PATH = "/Users/christinewu/Desktop/academic resources/Advanced Machine Learning/Homework Files/Homework1- Kaggle Competition/CSV Files/train.csv"
TEST_PATH = "/Users/christinewu/Desktop/academic resources/Advanced Machine Learning/Homework Files/Homework1- Kaggle Competition/CSV Files/test.csv"
SAMPLE_SUB_PATH = "/Users/christinewu/Desktop/academic resources/Advanced Machine Learning/Homework Files/Homework1- Kaggle Competition/CSV Files/sample_submission.csv"

TARGET = "Irrigation_Need"
ID_COL = "id"
RANDOM_STATE = 42
CV_FOLDS = 5
N_JOBS = -1

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

In [4]:
# 2) Split features / target
X = train_df.drop(columns=[TARGET])
y = train_df[TARGET].copy()

X = X.drop(columns=[ID_COL], errors="ignore")
X_test = test_df.drop(columns=[ID_COL], errors="ignore")

# Encode target labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [5]:
# 3) Identify numeric and categorical columns
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

In [6]:
# 4) Preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [7]:
# 5) Random Forest pipeline
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        class_weight="balanced_subsample"
    ))
])

param_dist = {
    "model__n_estimators": [150, 250, 350],
    "model__max_depth": [8, 12, 16, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", 0.6, 0.8],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_dist,
    n_iter=4,
    scoring="accuracy",
    cv=cv,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=1,
    refit=True
)

In [8]:
# 6) Train
search.fit(X, y_encoded)

print("Best Random Forest params:")
print(search.best_params_)
print("Best CV accuracy:", round(search.best_score_, 5))

best_model = search.best_estimator_

Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best Random Forest params:
{'model__n_estimators': 150, 'model__min_samples_split': 2, 'model__min_samples_leaf': 1, 'model__max_features': 0.6, 'model__max_depth': 16}
Best CV accuracy: 0.98545


In [9]:
# 7) Fit on full train and predict test
best_model.fit(X, y_encoded)
test_pred_encoded = best_model.predict(X_test)
test_pred = label_encoder.inverse_transform(test_pred_encoded)

In [10]:
# 8) Save submission
submission = sample_sub.copy()
submission[TARGET] = test_pred
submission.to_csv("submission_bagging.csv", index=False)

## Additional Bagging Model Comparison

To build on my Random Forest model, I added an Extra Trees model as a second bagging-style ensemble. To reduce runtime, I used 3-fold cross-validation and a moderate number of trees instead of a much larger ensemble.

In [15]:
import numpy as np
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, balanced_accuracy_score

In [12]:
# CV setup
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
balanced_scorer = make_scorer(balanced_accuracy_score)

### Fit a lighter Extra Trees model

In [16]:
et_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ExtraTreesClassifier(
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        class_weight="balanced",
        n_estimators=200,
        max_depth=12,
        min_samples_split=8,
        min_samples_leaf=3,
        max_features="sqrt"
    ))
])

et_cv_scores = cross_val_score(
    et_pipeline,
    X,
    y_encoded,
    cv=cv,
    scoring=balanced_scorer,
    n_jobs=N_JOBS
)

print("Extra Trees CV balanced accuracy scores:", np.round(et_cv_scores, 5))
print("Extra Trees mean CV balanced accuracy:", round(et_cv_scores.mean(), 5))
print("Extra Trees std CV balanced accuracy:", round(et_cv_scores.std(), 5))

Extra Trees CV balanced accuracy scores: [0.88591 0.88703 0.88845]
Extra Trees mean CV balanced accuracy: 0.88713
Extra Trees std CV balanced accuracy: 0.00104


Exception ignored in: <function ResourceTracker.__del__ at 0x102dd5da0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x10284dda0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x102729da0>
Traceback (most recent call last

### Train Extra Trees on the full training data

In [ ]:
et_pipeline.fit(X, y_encoded)

et_test_pred_encoded = et_pipeline.predict(X_test)
et_test_pred = label_encoder.inverse_transform(et_test_pred_encoded)

submission_et = sample_sub.copy()
submission_et[TARGET] = et_test_pred
submission_et.to_csv("submission_extra_trees.csv", index=False)

submission_et.head()

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


Exception ignored in: <function ResourceTracker.__del__ at 0x103965da0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x102db9da0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x104fb5da0>
Traceback (most recent call last

### Bagging Results

In [18]:
bagging_results = pd.DataFrame({
    "Model": ["Random Forest", "Extra Trees"],
    "CV Balanced Accuracy": [
        round(search.best_score_, 5),   # your tuned Random Forest CV from earlier cells
        0.88713
    ],
    "CV Std": [
        np.nan,
        0.00104
    ],
    "Kaggle Score": [
        0.96034,
        0.88387
    ]
})

bagging_results

,Model,CV Balanced Accuracy,CV Std,Kaggle Score
0,Random Forest,0.98545,NaN,0.96034
1,Extra Trees,0.88713,0.00104,0.88387


## Bagging Discussion

In this notebook, I compared two bagging ensemble methods: Random Forest and Extra Trees. Both methods combine predictions from many decision trees, but Extra Trees adds more randomness to the splitting process than Random Forest.

The Random Forest model performed much better overall. Its Kaggle public score was 0.96034, while the Extra Trees model only scored 0.88387. The Extra Trees cross-validation balanced accuracy was also much lower at 0.88713, which showed that its weaker leaderboard performance was consistent with validation performance.

This means that the additional randomness in Extra Trees did not help for this dataset. Instead, it appears that the Random Forest model was better able to capture the important structure in the data while still generalizing well.

My takeaway is that bagging can work very well for this competition, but not every bagging-style tree ensemble performs equally well. In my work, Random Forest was clearly the stronger bagging model.

## Bagging Model Comparison

Comparing the two bagging models, Random Forest was the clear winner. Extra Trees ran quickly and provided a useful second comparison point, but its balanced accuracy and Kaggle score were both much worse than Random Forest.

This suggests that for this dataset, introducing more random splits hurt performance instead of helping it. Overall, Random Forest was the better bagging approach for my work.